# 🎙 Voice Studio — Colab Worker

**Pipeline:** `Demucs (разделение)` → `Whisper (транскрипция)`

1. Запусти `bash start.sh` на телефоне
2. Скопируй публичный URL и секрет ниже
3. **Runtime → Run all**

## ⚙️ Конфиг

In [ ]:
# ════════════════════════════════════
STUDIO_URL   = "https://your-tunnel.trycloudflare.com"  # из start.sh
COLAB_SECRET = "your_secret_here"                       # из .env
# ════════════════════════════════════

POLL_INTERVAL    = 10
MAX_ERRORS       = 5
DEMUCS_MODEL     = "htdemucs"   # htdemucs | htdemucs_ft | mdx_extra
WHISPER_MODEL    = "base"       # tiny | base | small | medium | large
WHISPER_LANGUAGE = None         # None=autodetect | 'ru' | 'en'
WORK_DIR         = "/content/vs_work"

# MP3 bitrate для отправки на телефон.
# 192k = ~6MB на трек, влезает в 100s лимит Cloudflare.
# Если интернет быстрый — можно поставить 320k.
UPLOAD_BITRATE = "192k"

STUDIO_URL = STUDIO_URL.rstrip("/")
print(f"Studio : {STUDIO_URL}")
print(f"Demucs : {DEMUCS_MODEL}")
print(f"Whisper: {WHISPER_MODEL}")
print(f"Upload : MP3 {UPLOAD_BITRATE}")

## 1. Установка

In [ ]:
import subprocess, sys

def run(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and r.returncode != 0:
        print(r.stderr[-600:])
        raise RuntimeError(f"Упало: {cmd[:60]}")
    return r.stdout

print("pip install demucs openai-whisper...")
run("pip install -q demucs openai-whisper")

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cuda":
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory // (1024**3)
    print(f"✓ GPU: {name} ({mem}GB)")
else:
    print("⚠️  GPU не найден → Runtime → Change runtime type → T4 GPU")

print(f"✓ device={DEVICE}")

## 2. Загрузка Whisper

In [ ]:
import whisper

WHISPER_CACHE = {}

def get_whisper(name):
    if name not in WHISPER_CACHE:
        print(f"Загружаю Whisper '{name}'...")
        WHISPER_CACHE[name] = whisper.load_model(name, device=DEVICE)
        print(f"✓ Whisper '{name}' готов")
    return WHISPER_CACHE[name]

get_whisper(WHISPER_MODEL)
print("✓ Готов")

## 3. Проверка соединения

In [ ]:
import requests

HEADERS = {
    "bypass-tunnel-reminder": "true",
    "User-Agent": "VoiceStudio-Colab/1.0",
}

def api(method, path, **kwargs):
    url = f"{STUDIO_URL}{path}"
    headers = {**HEADERS, **kwargs.pop("headers", {})}
    kwargs.setdefault("timeout", 30)
    r = requests.request(method, url, headers=headers, **kwargs)
    if not r.ok:
        try:    detail = r.json()
        except: detail = r.text[:300]
        raise requests.exceptions.HTTPError(
            f"{r.status_code} {r.reason} → {detail}", response=r
        )
    return r

try:
    h = api("GET", "/health").json()
    print(f"✓ Сервер: {h.get('status')} | uptime {h.get('uptime_sec',0)}s")
    jobs = api("GET", f"/api/jobs/pending?secret={COLAB_SECRET}").json()
    print(f"✓ Авторизация OK | Pending: {len(jobs)}")
    for j in jobs:
        print(f"  {j['type']:14s} {j['job_id'][:8]}... project={j['project_id'][:8]}...")
except requests.exceptions.HTTPError as e:
    print(f"✗ HTTP: {e}")
except requests.exceptions.ConnectionError:
    print(f"✗ Нет соединения с {STUDIO_URL}")
    print("  → start.sh запущен? URL правильный? Интернет есть?")
except Exception as e:
    print(f"✗ {e}")

## 4. Функции воркера

In [ ]:
import time, shutil, traceback, json, os
from pathlib import Path
from datetime import datetime

os.makedirs(WORK_DIR, exist_ok=True)


def log(msg, level="INFO"):
    icons = {"INFO": "ℹ️ ", "OK": "✓  ", "ERR": "✗  ", "WARN": "⚠️ "}
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {icons.get(level,'')} {msg}")


# ── Скачать файл с телефона ───────────────────────────────────

def download_file(url, dest):
    r = requests.get(url, headers=HEADERS, stream=True, timeout=180)
    r.raise_for_status()
    written = 0
    with open(dest, "wb") as f:
        for chunk in r.iter_content(256 * 1024):
            f.write(chunk)
            written += len(chunk)
    log(f"Скачано: {Path(dest).name} ({written/1_048_576:.1f}MB)", "OK")


# ── Сжатие перед отправкой (обход Cloudflare 524) ────────────
# Cloudflare quick tunnel режет соединение через ~100с.
# WAV 37MB при 3Mbps = 100с (ровно на грани, не успевает).
# MP3 192k ~6MB = 16с (успевает с запасом).

def to_mp3(wav_path, bitrate=None):
    bitrate  = bitrate or UPLOAD_BITRATE
    mp3_path = wav_path.with_suffix(".mp3")
    cmd = f'ffmpeg -y -i "{wav_path}" -b:a {bitrate} "{mp3_path}" -loglevel error'
    r   = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 or not mp3_path.exists():
        log(f"MP3 не вышел, шлём WAV", "WARN")
        return wav_path, "audio/wav"
    w = wav_path.stat().st_size  / 1_048_576
    m = mp3_path.stat().st_size  / 1_048_576
    log(f"{wav_path.name}: {w:.1f}MB → {m:.1f}MB MP3 ({100*(1-m/w):.0f}% меньше)", "OK")
    return mp3_path, "audio/mpeg"


# ── Demucs ────────────────────────────────────────────────────

def run_demucs(input_path, job_dir, model=None):
    model   = model or DEMUCS_MODEL
    out_dir = job_dir / "separated"
    out_dir.mkdir(exist_ok=True)

    log(f"Demucs ({model})...")
    t0  = time.time()
    cmd = (
        f'python -m demucs --name "{model}" --two-stems vocals '
        f'--out "{out_dir}" "{input_path}"'
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Demucs failed:\n{r.stderr[-600:]}")

    log(f"Demucs готов за {time.time()-t0:.1f}с", "OK")

    vocals  = list(out_dir.rglob("vocals.wav"))
    novoice = list(out_dir.rglob("no_vocals.wav"))
    if not vocals or not novoice:
        files = [str(p) for p in out_dir.rglob("*")]
        raise FileNotFoundError(f"Demucs не создал файлы:\n" + "\n".join(files))

    vocal_dest  = job_dir / "vocal.wav"
    instru_dest = job_dir / "instrumental.wav"
    shutil.copy2(vocals[0],  vocal_dest)
    shutil.copy2(novoice[0], instru_dest)

    log(
        f"vocal={vocal_dest.stat().st_size/1_048_576:.1f}MB  "
        f"instru={instru_dest.stat().st_size/1_048_576:.1f}MB",
        "OK"
    )
    return vocal_dest, instru_dest


def upload_separation(job_id, vocal_path, instru_path):
    log("Сжимаю в MP3...")
    v_file, v_type = to_mp3(vocal_path)
    i_file, i_type = to_mp3(instru_path)

    total_mb = (v_file.stat().st_size + i_file.stat().st_size) / 1_048_576
    log(f"Загружаю {total_mb:.1f}MB на телефон...")

    with open(v_file, "rb") as vf, open(i_file, "rb") as inf:
        r = api(
            "POST",
            f"/api/jobs/{job_id}/complete?secret={COLAB_SECRET}",
            files={
                "vocal_file":        (v_file.name, vf,  v_type),
                "instrumental_file": (i_file.name, inf, i_type),
            },
            timeout=120,
        )
    log(f"Принято: {r.json()}", "OK")


# ── Whisper ───────────────────────────────────────────────────

def run_whisper(audio_path, model_name=None):
    model_name = model_name or WHISPER_MODEL
    model      = get_whisper(model_name)

    log(f"Whisper ({model_name})...")
    t0   = time.time()
    opts = {"task": "transcribe", "verbose": False}
    if WHISPER_LANGUAGE:
        opts["language"] = WHISPER_LANGUAGE

    result = model.transcribe(str(audio_path), **opts)
    segs   = result.get("segments", [])
    log(
        f"Whisper готов за {time.time()-t0:.1f}с | "
        f"lang={result.get('language')} | segs={len(segs)}",
        "OK"
    )
    return result


def upload_transcription(job_id, result):
    log("Отправляю транскрипцию...")
    segs = [
        {
            "id":          s.get("id"),
            "start":       round(float(s.get("start", 0)), 3),
            "end":         round(float(s.get("end",   0)), 3),
            "text":        s.get("text", "").strip(),
            "avg_logprob": round(float(s.get("avg_logprob", 0)), 4),
        }
        for s in result.get("segments", [])
        if s.get("text", "").strip()
    ]
    body = {
        "segments_json": json.dumps(segs, ensure_ascii=False),
        "language":      result.get("language", "ru"),
        "model_used":    WHISPER_MODEL,
    }
    r    = api(
        "POST",
        f"/api/jobs/{job_id}/complete-transcription?secret={COLAB_SECRET}",
        json=body, timeout=60,
    )
    d = r.json()
    log(f"Принято: lines={d.get('lines')} slicing={str(d.get('slicing_job_id',''))[:8]}", "OK")


# ── Обработка заданий ─────────────────────────────────────────

def process_separation(job):
    job_id     = job["job_id"]
    project_id = job["project_id"]
    job_dir    = Path(WORK_DIR) / job_id
    job_dir.mkdir(exist_ok=True)
    log(f"══ SEPARATION {job_id[:8]} ══")
    try:
        api("POST", f"/api/jobs/{job_id}/claim?secret={COLAB_SECRET}")

        # Строим URL через STUDIO_URL — download_url сервера может быть 0.0.0.0
        url = f"{STUDIO_URL}/api/projects/{project_id}/assets/original"
        download_file(url, job_dir / "original.wav")

        sep_model = job.get("input_params", {}).get("demucs_model", DEMUCS_MODEL)
        vocal, instru = run_demucs(job_dir / "original.wav", job_dir, model=sep_model)
        upload_separation(job_id, vocal, instru)

        log(f"══ SEPARATION {job_id[:8]} DONE ✓ ══", "OK")
        return True
    except Exception as e:
        log(f"══ SEPARATION ERROR: {e} ══", "ERR")
        traceback.print_exc()
        return False
    finally:
        shutil.rmtree(job_dir, ignore_errors=True)


def process_transcription(job):
    job_id     = job["job_id"]
    project_id = job["project_id"]
    job_dir    = Path(WORK_DIR) / job_id
    job_dir.mkdir(exist_ok=True)
    log(f"══ TRANSCRIPTION {job_id[:8]} ══")
    try:
        api("POST", f"/api/jobs/{job_id}/claim?secret={COLAB_SECRET}")

        url = f"{STUDIO_URL}/api/projects/{project_id}/assets/vocal"
        vocal_path = job_dir / "vocal.wav"
        download_file(url, vocal_path)

        wh_model = job.get("input_params", {}).get("whisper_model", WHISPER_MODEL)
        result   = run_whisper(vocal_path, model_name=wh_model)
        upload_transcription(job_id, result)

        log(f"══ TRANSCRIPTION {job_id[:8]} DONE ✓ ══", "OK")
        return True
    except Exception as e:
        log(f"══ TRANSCRIPTION ERROR: {e} ══", "ERR")
        traceback.print_exc()
        return False
    finally:
        shutil.rmtree(job_dir, ignore_errors=True)


def process_job(job):
    t = job.get("type")
    if t == "separation":    return process_separation(job)
    if t == "transcription": return process_transcription(job)
    log(f"Неизвестный тип: {t}", "WARN")
    return False


print("✓ Функции загружены")

## 5. 🚀 Основной цикл
Остановить: **Runtime → Interrupt execution**

In [ ]:
log(f"Воркер запущен | Demucs={DEMUCS_MODEL} | Whisper={WHISPER_MODEL} | poll={POLL_INTERVAL}s | upload=MP3 {UPLOAD_BITRATE}")
print("─" * 60)

err_streak = 0
stats = {"separation": 0, "transcription": 0, "failed": 0}

try:
    while True:
        try:
            jobs = api("GET", f"/api/jobs/pending?secret={COLAB_SECRET}").json()
            err_streak = 0

            if not jobs:
                print(f"[{datetime.now().strftime('%H:%M:%S')}]   нет заданий...", end="\r")
            else:
                print()
                log(f"Заданий: {len(jobs)}")
                for job in jobs:
                    ok = process_job(job)
                    key = job["type"] if job["type"] in stats else "failed"
                    stats[key if ok else "failed"] += 1
                log(f"sep={stats['separation']} trans={stats['transcription']} fail={stats['failed']}")
                print("─" * 60)

        except KeyboardInterrupt:
            raise
        except requests.exceptions.ConnectionError:
            err_streak += 1
            print()
            log(f"Нет связи ({err_streak}/{MAX_ERRORS})", "WARN")
            if err_streak >= MAX_ERRORS:
                log("Стоп — слишком много ошибок", "ERR"); break
        except Exception as e:
            err_streak += 1
            print()
            log(f"Ошибка: {e} ({err_streak}/{MAX_ERRORS})", "ERR")
            traceback.print_exc()
            if err_streak >= MAX_ERRORS:
                break

        time.sleep(POLL_INTERVAL)

except KeyboardInterrupt:
    print()
    log("Остановлен вручную.")

print("─" * 60)
log(f"Итого: sep={stats['separation']} trans={stats['transcription']} fail={stats['failed']}")

---
## 🔧 Утилиты

In [ ]:
# Очередь + метрики
jobs = api("GET", f"/api/jobs/pending?secret={COLAB_SECRET}").json()
print(f"Pending: {len(jobs)}")
for j in jobs:
    print(f"  {j['type']:14s} {j['job_id'][:8]}... project={j['project_id'][:8]}...")

m = api("GET", "/api/metrics").json()
print(f"\nUptime : {m['uptime']}")
print(f"Storage: {m['storage']['used_mb']}MB / free {m['storage']['free_mb']}MB")
print(f"Jobs   : {m['jobs']}")

In [ ]:
# Очистить рабочую папку
shutil.rmtree(WORK_DIR, ignore_errors=True)
os.makedirs(WORK_DIR)
print(run("df -h /content"))
print(f"✓ {WORK_DIR} очищен")